# Coins: contour saliency weighted by circularity

## Goal

This tutorial builds a hierarchy $H_{\mathrm{circ}}$ **without constructing a shape space**. A max-tree is built from the coins image. Area-based extinction values select cutoff nodes, and the circularity of those nodes weights their complete contours:

$$w(e)=\max\{\operatorname{circ}(n)\mid e\in\partial R_n\}.$$

The edge map $w$ implicitly represents $H_{\mathrm{circ}}=QFZ(G,w)$. The QFZ tree need not be materialized to display the map, extract cuts, or obtain threshold partitions.

`ShapeSpaceSaliency.projectContourScores` is used only as a generic node-score-to-contour projector. This notebook does not call `computeExtinctionValues` and does not build a second component tree on the parent-child graph. The extinction traversal is based on Silva and Lotufo, [*Efficient computation of new extinction values from extended component tree*](https://doi.org/10.1016/j.patrec.2010.07.019), *Pattern Recognition Letters* 32(1), 79–90, 2011. The QFZ/saliency correspondence is defined by Cousty, Najman, Kenmochi, and Guimarães, [*Hierarchical segmentations with graphs: quasi-flat zones, minimum spanning trees, and saliency maps*](https://doi.org/10.1007/s10851-017-0768-7), *Journal of Mathematical Imaging and Vision* 60(4), 479–502, 2018. See the [code-to-paper correspondence](../docs/saliency.md#primary-references-and-implementation-correspondence).

## Setup

This notebook assumes that the `notebooks` environment described in `README.md` is already active. The first code cell imports the installed package directly with `import mmcfilters`; no build-tree loader or installation cell is used.

In [ ]:
from __future__ import annotations

import mmcfilters
from IPython.display import display
from matplotlib.collections import LineCollection
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import connected_components
from skimage import data

%matplotlib inline
plt.rcParams.update({"figure.dpi": 105, "axes.titlesize": 11})
print(f"mmcfilters: {mmcfilters.__version__}")


### Parameters and image

`MIN_AREA` removes small details before the analysis. `TOP_K` selects the most persistent finite area extrema. The root-associated dominant extremum is excluded because its sentinel does not represent a coin. The value 24 approximately matches the visible number of coins. Radius 1.5 selects 8-connectivity.

In [ ]:
MIN_AREA = 500
TOP_K = 24
ADJACENCY_RADIUS = 1.5
QFZ_QUANTILES = (0.50, 0.75, 0.90)

image = np.ascontiguousarray(data.coins(), dtype=np.uint8)
num_rows, num_cols = image.shape

plt.figure(figsize=(6.4, 5.0))
plt.imshow(image, cmap="gray", vmin=0, vmax=255)
plt.title("Input image: skimage.data.coins()")
plt.axis("off")
plt.show()

print(f"domain: {num_rows} x {num_cols}; pixels: {image.size}")

## Steps

### 1. Simplify the image by area

Following the preparation used in `SimpleExamples.ipynb`, we build a max-tree, remove components with area at most `MIN_AREA` through the subtractive rule, and build a new max-tree from the filtered image.

In [ ]:
initial_tree = mmcfilters.MorphologicalTreeFactory.createMaxTree(
    image,
    radius=ADJACENCY_RADIUS,
)
initial_area = mmcfilters.Attribute.computeSingleTopologyAttribute(
    initial_tree,
    mmcfilters.Attribute.AREA,
)
filtered_image = np.ascontiguousarray(
    mmcfilters.AttributeFilters(initial_tree).filteringSubtractiveRule(
        initial_area > MIN_AREA
    ),
    dtype=np.uint8,
)

tree = mmcfilters.MorphologicalTreeFactory.createMaxTree(
    filtered_image,
    radius=ADJACENCY_RADIUS,
)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.8), constrained_layout=True)
axes[0].imshow(image, cmap="gray", vmin=0, vmax=255)
axes[0].set_title("Input")
axes[1].imshow(filtered_image, cmap="gray", vmin=0, vmax=255)
axes[1].set_title(f"Area-filtered > {MIN_AREA}")
for axis in axes:
    axis.axis("off")
plt.show()

print(f"live nodes in final max-tree: {len(tree.aliveNodeIds)}")
print(f"leaves in final max-tree: {tree.numLeafNodes}")

### 2. Select cutoff nodes by area extinction

Area is an increasing max-tree attribute and can drive the classical `ExtinctionValues` algorithm. Circularity does not affect selection; it is used only to weight the selected contours.

In [ ]:
area = np.ascontiguousarray(
    mmcfilters.Attribute.computeSingleTopologyAttribute(
        tree,
        mmcfilters.Attribute.AREA,
    ),
    dtype=np.float32,
)
circularity = np.ascontiguousarray(
    mmcfilters.Attribute.computeSingleTopologyAttribute(
        tree,
        mmcfilters.Attribute.CIRCULARITY,
    ),
    dtype=np.float32,
)

extinction_values = mmcfilters.ExtinctionValues(tree, area)
ranked_records = sorted(
    extinction_values.getRegionalExtrema(),
    key=lambda record: (-float(record[2]), int(record[0]), int(record[1])),
)
float32_max = np.finfo(np.float32).max
finite_records = [
    record for record in ranked_records
    if float(record[2]) < float32_max
]
selected_records = finite_records[: min(TOP_K, len(finite_records))]

selection_rows = []
for rank, (leaf_id, cutoff_node_id, extinction) in enumerate(selected_records, start=1):
    extinction = float(extinction)
    dominant = extinction >= float32_max
    selection_rows.append({
        "rank": rank,
        "leaf": int(leaf_id),
        "cutoffNode": int(cutoff_node_id),
        "areaExtinction": np.nan if dominant else extinction,
        "dominant": dominant,
        "circularity": float(circularity[int(cutoff_node_id)]),
    })

selection_frame = pd.DataFrame(selection_rows)
display(selection_frame.head(12))
print(f"selected finite extrema: {len(selected_records)} / {len(finite_records)}")
print(f"excluded dominant extremum: {len(ranked_records) - len(finite_records)}")

### 3. Confirm that circularity is not a max-tree valuation

A hierarchy valuation must satisfy `valuation(parent) >= valuation(child)`. Raw circularity normally violates this condition, so it is not passed directly to `HierarchySaliencyMap.computeSaliencyEdgeMap`.

In [ ]:
monotonicity_violations = []
for parent_id in tree.aliveNodeIds:
    parent_id = int(parent_id)
    for child_id in tree.getChildren(parent_id):
        child_id = int(child_id)
        if circularity[parent_id] < circularity[child_id]:
            monotonicity_violations.append((parent_id, child_id))

print(
    "parent-child relations that violate circularity(parent) >= circularity(child): "
    f"{len(monotonicity_violations)}"
)

try:
    mmcfilters.HierarchySaliencyMapValidation.validateHierarchyValuation(
        tree,
        circularity,
        nonnegative=True,
    )
    validation_message = "Circularity was accepted as a hierarchy valuation."
except ValueError as error:
    validation_message = f"Classical validation rejected it as expected: {error}"

print(validation_message)
assert monotonicity_violations, "Circularity was expected to be non-monotone for this image."

### 4. Weight complete contours by circularity

Only selected cutoff nodes receive positive circularity scores. When two extrema share a cutoff node, the larger value is retained. Projection assigns each image edge the maximum score among all contours that contain it.

In [ ]:
node_contour_scores = np.zeros(tree.numInternalNodeSlots, dtype=np.float32)
for _leaf_id, cutoff_node_id, _extinction in selected_records:
    cutoff_node_id = int(cutoff_node_id)
    node_contour_scores[cutoff_node_id] = max(
        node_contour_scores[cutoff_node_id],
        circularity[cutoff_node_id],
    )

circularity_edge_map = mmcfilters.ShapeSpaceSaliency.projectContourScores(
    tree,
    node_contour_scores,
)
edge_values = np.asarray(circularity_edge_map["values"])
positive_edge_values = edge_values[edge_values > 0]

assert positive_edge_values.size > 0
assert np.all(np.isfinite(edge_values))
assert np.all(edge_values >= 0)
assert edge_values.max() <= node_contour_scores.max()

saliency_image = np.asarray(
    mmcfilters.HierarchySaliencyMapProjection.edgeMapToPixelImage(
        circularity_edge_map
    )
)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.8), constrained_layout=True)
axes[0].imshow(filtered_image, cmap="gray", vmin=0, vmax=255)
axes[0].set_title("Filtered image")
saliency_artist = axes[1].imshow(
    saliency_image,
    cmap="magma",
    vmin=0,
    vmax=float(edge_values.max()),
)
axes[1].set_title(r"Rasterization of $w$: circularity on contours")
for axis in axes:
    axis.axis("off")
fig.colorbar(saliency_artist, ax=axes[1], fraction=0.046, pad=0.04)
plt.show()

print(f"nodes with contour scores: {np.count_nonzero(node_contour_scores)}")
print(f"adjacency edges: {edge_values.size}")
print(f"edges with positive saliency: {positive_edge_values.size}")
print(f"maximum saliency: {float(edge_values.max()):.6f}")

The operation above did not compute extinctions on the parent-child graph. Despite the class name, `projectContourScores` only applies the generic mapping `node scores -> maximum on contours`. The maximum is essential: sequential writes to pixels depend on node order and do not represent overlapping contours formally.

### 5. Compare with the former sequential rasterization

The former example wrote circularity directly to every contour pixel, so the last visited node won on shared pixels. We repeat that write in both orders. Maximum aggregation is order-independent, but remains a pixel raster; it must not be compared pixel-by-pixel with the formal edge map without stating the edge-to-pixel aggregation.

In [ ]:
legacy_contours = mmcfilters.ContourComputation.extraction(tree)


def legacy_pixel_contour_map(records, reverse=False, use_maximum=False):
    output = np.zeros(filtered_image.size, dtype=np.float32)
    ordered_records = list(reversed(records)) if reverse else list(records)
    for _leaf_id, cutoff_node_id, _extinction in ordered_records:
        cutoff_node_id = int(cutoff_node_id)
        pixels = np.fromiter(
            legacy_contours.getContour(cutoff_node_id),
            dtype=np.int64,
        )
        value = circularity[cutoff_node_id]
        if use_maximum:
            np.maximum.at(output, pixels, value)
        else:
            output[pixels] = value
    return output.reshape(filtered_image.shape)


legacy_forward = legacy_pixel_contour_map(selected_records)
legacy_reverse = legacy_pixel_contour_map(selected_records, reverse=True)
legacy_maximum = legacy_pixel_contour_map(selected_records, use_maximum=True)
legacy_maximum_reverse = legacy_pixel_contour_map(
    selected_records,
    reverse=True,
    use_maximum=True,
)

order_difference = np.abs(legacy_forward - legacy_reverse)
order_dependent_pixels = np.count_nonzero(order_difference)
assert order_dependent_pixels > 0
np.testing.assert_array_equal(legacy_maximum, legacy_maximum_reverse)

fig, axes = plt.subplots(1, 3, figsize=(14, 4.2), constrained_layout=True)
axes[0].imshow(legacy_forward, cmap="magma", vmin=0, vmax=float(circularity.max()))
axes[0].set_title("Write in ranking order")
axes[1].imshow(legacy_reverse, cmap="magma", vmin=0, vmax=float(circularity.max()))
axes[1].set_title("Write in reverse order")
axes[2].imshow(order_difference, cmap="inferno", vmin=0)
axes[2].set_title(f"Difference: {order_dependent_pixels} pixels")
for axis in axes:
    axis.axis("off")
plt.show()

print(f"pixels affected by write order: {order_dependent_pixels}")
print("maximum raster is order-independent: yes")

### 6. Inspect saliency-map cuts

A cut at level $\lambda$ keeps edges with $w(e)\geq\lambda$ as contours. Quantiles of positive values provide three illustrative, data-dependent levels.

In [ ]:
def edge_segments(edge_dictionary):
    sources = np.asarray(edge_dictionary["sources"], dtype=np.int64)
    targets = np.asarray(edge_dictionary["targets"], dtype=np.int64)
    cols = int(edge_dictionary["numCols"])
    source_xy = np.column_stack((sources % cols, sources // cols))
    target_xy = np.column_stack((targets % cols, targets // cols))
    return np.stack((source_xy, target_xy), axis=1)


cut_thresholds = np.unique(
    np.quantile(positive_edge_values, QFZ_QUANTILES)
)

fig, axes = plt.subplots(
    1,
    len(cut_thresholds),
    figsize=(5 * len(cut_thresholds), 4.4),
    constrained_layout=True,
)
axes = np.atleast_1d(axes)
cut_sizes = []
for axis, threshold in zip(axes, cut_thresholds):
    cut = mmcfilters.HierarchySaliencyMapProjection.thresholdCut(
        circularity_edge_map,
        float(threshold),
    )
    segments = edge_segments(cut)
    cut_sizes.append(len(segments))
    axis.imshow(filtered_image, cmap="gray", vmin=0, vmax=255)
    axis.add_collection(
        LineCollection(segments, colors="#ffcc00", linewidths=0.55, alpha=0.9)
    )
    axis.set_title(f"λ = {threshold:.3f}\n{len(segments)} contour edges")
    axis.set_xlim(-0.5, num_cols - 0.5)
    axis.set_ylim(num_rows - 0.5, -0.5)
    axis.axis("off")
plt.show()

### 7. Recover QFZ partitions of $H_{\mathrm{circ}}$

At level $\lambda$, pixels are connected through edges with $w(e)<\lambda$. Their connected components are the quasi-flat zones. As $\lambda$ increases, components only merge, so the partitions form a hierarchy.

In [ ]:
def qfz_partition(edge_dictionary, threshold):
    sources = np.asarray(edge_dictionary["sources"], dtype=np.int64)
    targets = np.asarray(edge_dictionary["targets"], dtype=np.int64)
    values = np.asarray(edge_dictionary["values"])
    rows = int(edge_dictionary["numRows"])
    cols = int(edge_dictionary["numCols"])

    active = values < threshold
    active_sources = sources[active]
    active_targets = targets[active]
    graph_rows = np.concatenate((active_sources, active_targets))
    graph_cols = np.concatenate((active_targets, active_sources))
    graph = coo_matrix(
        (np.ones(graph_rows.size, dtype=np.uint8), (graph_rows, graph_cols)),
        shape=(rows * cols, rows * cols),
    ).tocsr()
    component_count, labels = connected_components(
        graph,
        directed=False,
        return_labels=True,
    )
    return component_count, labels.reshape(rows, cols)


def is_refinement(fine_labels, coarse_labels):
    pairs = np.unique(
        np.column_stack((fine_labels.ravel(), coarse_labels.ravel())),
        axis=0,
    )
    return pairs.shape[0] == np.unique(pairs[:, 0]).size


qfz_partitions = [
    qfz_partition(circularity_edge_map, float(threshold))
    for threshold in cut_thresholds
]

for (_fine_count, fine), (_coarse_count, coarse) in zip(
    qfz_partitions,
    qfz_partitions[1:],
):
    assert is_refinement(fine, coarse)

fig, axes = plt.subplots(
    1,
    len(qfz_partitions),
    figsize=(5 * len(qfz_partitions), 4.4),
    constrained_layout=True,
)
axes = np.atleast_1d(axes)
for axis, threshold, (component_count, labels) in zip(
    axes,
    cut_thresholds,
    qfz_partitions,
):
    axis.imshow(labels, cmap="nipy_spectral", interpolation="nearest")
    axis.set_title(f"QFZ at λ = {threshold:.3f}\n{component_count} regions")
    axis.axis("off")
plt.show()

print(
    "region counts from finest to coarsest partition: "
    + " → ".join(str(count) for count, _labels in qfz_partitions)
)

## Checks

To show that $H_{\mathrm{circ}}$ is not merely the original max-tree with another scale, we also compute the least monotone majorant of the selected scores and apply classical LCA projection. That result is valid on the max-tree but differs from maximum-on-contours.

In [ ]:
monotone_valuation = node_contour_scores.copy()
for node_id in tree.getPostOrderNodes():
    node_id = int(node_id)
    for child_id in tree.getChildren(node_id):
        monotone_valuation[node_id] = max(
            monotone_valuation[node_id],
            monotone_valuation[int(child_id)],
        )

mmcfilters.HierarchySaliencyMapValidation.validateHierarchyValuation(
    tree,
    monotone_valuation,
    nonnegative=True,
)
classic_edge_map = mmcfilters.HierarchySaliencyMap.computeSaliencyEdgeMap(
    tree,
    monotone_valuation,
)
classic_values = np.asarray(classic_edge_map["values"])

np.testing.assert_array_equal(
    circularity_edge_map["sources"],
    classic_edge_map["sources"],
)
np.testing.assert_array_equal(
    circularity_edge_map["targets"],
    classic_edge_map["targets"],
)
different_edges = np.count_nonzero(edge_values != classic_values)
assert different_edges > 0

classic_image = np.asarray(
    mmcfilters.HierarchySaliencyMapProjection.edgeMapToPixelImage(
        classic_edge_map
    )
)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.8), constrained_layout=True)
axes[0].imshow(saliency_image, cmap="magma", vmin=0, vmax=float(edge_values.max()))
axes[0].set_title(r"$H_{circ}$: maximum on contours")
axes[1].imshow(classic_image, cmap="magma", vmin=0, vmax=float(edge_values.max()))
axes[1].set_title("Original max-tree: majorant + LCA")
for axis in axes:
    axis.axis("off")
plt.show()

print(f"edges with different values between constructions: {different_edges}")
print("QFZ partitions verified as nested: yes")
print("second shape-space tree constructed: no")

## Conclusions and next steps

- Raw circularity is not a monotone valuation of the coins max-tree.
- Weighting complete contours and resolving overlaps by maximum produces the edge map $w$ representing $H_{\mathrm{circ}}$.
- The sampled QFZ partitions are nested without materializing a second tree.
- A monotone majorant followed by LCA projection produces a different hierarchy and edge map.
- An explicit $H_{\mathrm{circ}}$ tree would require a `QFZ(G, w)`/MST construction from `EdgeSaliencyMap`.
- Shape space is needed only when raw circularity is replaced by the persistence of its maxima or minima on the parent-child graph.